<a href="https://colab.research.google.com/github/msatyasambhav-beep/Machine-Learning/blob/main/HPO_BreastCancer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import accuracy_score


In [2]:
dataset_url = "https://raw.githubusercontent.com/apogiatzis/breast-cancer-azure-ml-notebook/master/breast-cancer-data.csv"
dataset = pd.read_csv(dataset_url)

In [3]:
dataset.head(5)

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [5]:
#Distribution of y-variable
dataset.diagnosis.value_counts()

,count
diagnosis,
B,357
M,212


In [7]:
dataset.diagnosis.value_counts()/len(dataset)*100

,count
diagnosis,
B,62.741652
M,37.258348


In [8]:
dataset.shape

(569, 33)

In [9]:
dataset['diagnosis'] = dataset['diagnosis'].map({'M': 1, 'B':0})

In [10]:
dataset.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,1,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,1,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,1,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,1,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [11]:
dataset.diagnosis.value_counts()

,count
diagnosis,
0,357
1,212


In [12]:
#X and y

X = dataset.iloc[:, [2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20]]
y = dataset.diagnosis.values

####Train Test Split

In [13]:
#Train-Test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size = 0.2,random_state=0)

In [14]:
len(X_train)

455

In [15]:
len(X_test)

114

###Feature Scaling

In [ ]:
#Standardization
#Normalization
from sklearn.preprocessing import StandardScaler
sc= StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

####Model Building(RF)

In [16]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier()
model.fit(X_train,y_train)

RandomForestClassifier()

In [18]:
#Predicting the test set values
y_pred = model.predict(X_test)


In [19]:
#Metrics

from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
result = accuracy_score(y_test, y_pred)
print("Accuracy of the base RF model is: ", round(result*100,2))

Accuracy of the base RF model is:  93.86


In [23]:
cf = confusion_matrix(y_test, y_pred)
print("CM is: ")
print(cf)

CM is: 
[[63  4]
 [ 3 44]]


In [22]:
cr = classification_report(y_test, y_pred)
print("Classification Report is: ")
print(cr)

Classification Report is: 
              precision    recall  f1-score   support

           0       0.95      0.94      0.95        67
           1       0.92      0.94      0.93        47

    accuracy                           0.94       114
   macro avg       0.94      0.94      0.94       114
weighted avg       0.94      0.94      0.94       114



####Manual HPO

In [24]:
n_estimators_list = [1,2,3,10,50,100,200]

for estim_list in n_estimators_list:
  model = RandomForestClassifier(n_estimators=estim_list)
  model.fit(X_train,y_train)
  y_pred = model.predict(X_test)
  result = accuracy_score(y_test, y_pred)
  print("\n Estimator Value",estim_list)
  print("Accuracy is", result)


 Estimator Value 1
Accuracy is 0.9210526315789473

 Estimator Value 2
Accuracy is 0.8947368421052632

 Estimator Value 3
Accuracy is 0.9298245614035088

 Estimator Value 10
Accuracy is 0.9298245614035088

 Estimator Value 50
Accuracy is 0.9385964912280702

 Estimator Value 100
Accuracy is 0.9385964912280702

 Estimator Value 200
Accuracy is 0.9385964912280702


In [25]:
leaf_size = [1,2,3,4,5,10]

for i in leaf_size:
  model = RandomForestClassifier(n_estimators=10,min_samples_leaf=i)
  model.fit(X_train,y_train)
  y_pred = model.predict(X_test)
  result = accuracy_score(y_test, y_pred)
  print("\n Estimator Value",i)
  print("Accuracy is", result)



 Estimator Value 1
Accuracy is 0.9385964912280702

 Estimator Value 2
Accuracy is 0.9385964912280702

 Estimator Value 3
Accuracy is 0.9473684210526315

 Estimator Value 4
Accuracy is 0.9385964912280702

 Estimator Value 5
Accuracy is 0.9122807017543859

 Estimator Value 10
Accuracy is 0.9298245614035088


###Random Search CV

In [35]:
n_estimators = [int(x) for x in np.linspace(start=100, stop=1000, num=10)]
max_depth = [int(x) for x in np.linspace(start=10, stop=110, num=11)]

print(n_estimators)
print(max_depth)

[100, 200, 300, 400, 500, 600, 700, 800, 900, 1000]
[10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110]


In [36]:
from sklearn.model_selection import RandomizedSearchCV

n_estimators = [int(x) for x in np.linspace(start=100, stop=1000, num=10)]
max_depth = [int(x) for x in np.linspace(start=10, stop=110, num=11)]
min_samples_split = [1,2,4,10,20,50,100]
min_samples_leaf=[2,3,4,5,8,20,50,100,200]
bootstrap = [True, False]

#Create a random grid
random_grid = {'n_estimators': n_estimators,
               'max_depth': max_depth,
               'min_samples_split': min_samples_split,
               'bootstrap': bootstrap,
               'min_samples_leaf':min_samples_split}

#####On each iteration ,the algorithm will choose a differenr combination of the features. Altogether , there are 15,400 combination .However the benefit of random search is that we are not trying every combination, but selecting at random to sample a wide range of values

In [40]:
#Use the random grid to search for best hyper parameters
#First create the base model to tune

rf = RandomForestClassifier()
#Random search of parameters using 3 fold cross validation
rf_random = RandomizedSearchCV(estimator=rf, param_distributions=random_grid, n_iter=100, cv=3, verbose=2, n_jobs=-1)
##n-jobs -1 means running it on cpu
#Fit the random search model
rf_random.fit(X_train, y_train)

Fitting 3 folds for each of 100 candidates, totalling 300 fits


/usr/local/lib/python3.13/dist-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
36 fits failed out of a total of 300.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
36 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/sklearn/base.py", line 1382, in wrapper
    estimator._validate_params()
    ~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "/usr/local/lib/python3.13/dist-packages/sklearn/base.py", line 436, in _validate_params
    validate_pa

RandomizedSearchCV(cv=3, estimator=RandomForestClassifier(), n_iter=100,
                   n_jobs=-1,
                   param_distributions={'bootstrap': [True, False],
                                        'max_depth': [10, 20, 30, 40, 50, 60,
                                                      70, 80, 90, 100, 110],
                                        'min_samples_leaf': [1, 2, 4, 10, 20,
                                                             50, 100],
                                        'min_samples_split': [1, 2, 4, 10, 20,
                                                              50, 100],
                                        'n_estimators': [100, 200, 300, 400,
                                                         500, 600, 700, 800,
                                                         900, 1000]},
                   verbose=2)

In [41]:
def evaluate(model, test_features, test_labels):
  predictions = model.predict(test_features)
  accuracy = accuracy_score(test_labels, predictions)
  print('Model Performance')
  print('Accuracy = {:0.2f}%.'.format(accuracy))
  return accuracy

In [42]:
base_model = RandomForestClassifier(n_estimators=5, random_state=42)
base_model.fit(X_train, y_train)
base_accuracy = evaluate(base_model, X_test, y_test)

Model Performance
Accuracy = 0.93%.


In [43]:
best_random = rf_random.best_estimator_
print(best_random)

RandomForestClassifier(bootstrap=False, max_depth=50, min_samples_split=4,
                       n_estimators=1000)


In [44]:
random_accuracy = evaluate(best_random, X_test, y_test)

Model Performance
Accuracy = 0.94%.


In [45]:
print('Improvement of {:0.2f}%.'.format(100*(random_accuracy - base_accuracy) / base_accuracy))

Improvement of 0.94%.


##Grid Search CV

In [46]:
from sklearn.model_selection import GridSearchCV
# Create the parameter grid based on the results of random search
param_grid = {
    'bootstrap': [True],
    'max_depth': [75, 80, 85, 90, 95, 100],
    'max_features': [2, 3, 4, 5],
    'min_samples_leaf': [1, 2, 3],
    'min_samples_split': [8, 10, 12],
    'n_estimators': [200, 250, 270, 300, 350, 400, 450, 500]
}
# Create a based model
rf_gd = RandomForestClassifier()
# Instantiate the grid search model
grid_search = GridSearchCV(estimator = rf_gd, param_grid = param_grid,
                          cv = 3, n_jobs = -1)

In [47]:
#Fit the grid  search to the data
grid_search.fit(X_train, y_train)
grid_search.best_params_

{'bootstrap': True,
 'max_depth': 80,
 'max_features': 3,
 'min_samples_leaf': 1,
 'min_samples_split': 8,
 'n_estimators': 270}

In [48]:
best_grid = grid_search.best_estimator_
grid_accuracy = evaluate(best_grid, X_test, y_test)

Model Performance
Accuracy = 0.94%.


In [49]:
print('Improvement of {:0.2f}%.'.format(100*(grid_accuracy - base_accuracy) / base_accuracy))

Improvement of 0.94%.


In [50]:
best_grid

RandomForestClassifier(max_depth=80, max_features=3, min_samples_split=8,
                       n_estimators=270)